# LoRA Fine-Tune MegaDescriptor (Colab + Drive)

Fine-tune **MegaDescriptor-L-384** with **LoRA + triplet/ArcFace** on Amvrakikos + Reunion using **paired augmentation** (flip, crop, blur), then evaluate zero-shot opposite-side re-ID on held-out **Zakynthos**.

## Before you run
1. Open this notebook in **Google Colab**.
2. Set **Runtime → Change runtime type → GPU** (T4 or better).
3. Put datasets under Drive:
   ```
   MyDrive/SeaTurtle/
     AmvrakikosTurtles/
     ReunionTurtles/
     ZakynthosTurtles/
   ```
4. Run cells top to bottom. Cell 1 always pulls the latest repo commit and clears cached imports.
5. **Zakynthos is sealed** until the final eval cell — recipe selection uses source-only holdouts.
6. Checkpoints land in `MyDrive/SeaTurtle/checkpoints/lora_megadescriptor_opp/`. Metrics CSVs and bootstrap CIs are saved alongside.

In [15]:
import os
import shutil
import sys
import subprocess

REPO_URL = 'https://github.com/abui-am/side-matching.git'
CLONE_DIR = '/content/sides-matching'

def assert_colab_gpu():
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError(
            'This notebook is Colab-only. Open it in Google Colab with a GPU runtime.'
        ) from exc
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            'CUDA GPU is required. In Colab choose Runtime > Change runtime type > GPU.'
        )
    print(f'Colab GPU: {torch.cuda.get_device_name(0)}')
    return torch

def find_repo_root():
    path = os.path.abspath(os.getcwd())
    for _ in range(6):
        if os.path.isdir(os.path.join(path, 'sides_matching')):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            break
        path = parent
    for candidate in [CLONE_DIR, '/content']:
        if os.path.isdir(os.path.join(candidate, 'sides_matching')):
            return candidate
    return None

def _purge_sides_matching_modules():
    for name in list(sys.modules):
        if name == 'sides_matching' or name.startswith('sides_matching.'):
            del sys.modules[name]

def ensure_repo_root():
    """Clone or hard-reset to origin/main so Colab never keeps a stale checkout."""
    root = find_repo_root()
    if root is None:
        print(f'Cloning {REPO_URL} -> {CLONE_DIR}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, CLONE_DIR])
        root = CLONE_DIR
    elif os.path.isdir(os.path.join(root, '.git')):
        print(f'Updating repo at {root} to origin/main...')
        try:
            subprocess.check_call(['git', '-C', root, 'fetch', '--depth', '1', 'origin', 'main'])
            subprocess.check_call(['git', '-C', root, 'reset', '--hard', 'origin/main'])
        except subprocess.CalledProcessError:
            print('git update failed; re-cloning...')
            shutil.rmtree(root)
            subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, CLONE_DIR])
            root = CLONE_DIR
    else:
        print(f'No .git at {root}; re-cloning into {CLONE_DIR}')
        if os.path.isdir(CLONE_DIR):
            shutil.rmtree(CLONE_DIR)
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, CLONE_DIR])
        root = CLONE_DIR
    _purge_sides_matching_modules()
    commit = subprocess.check_output(
        ['git', '-C', root, 'rev-parse', '--short', 'HEAD'], text=True
    ).strip()
    print(f'Repo commit: {commit}')
    return root

def pip_install(*packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *packages]
    print('>', ' '.join(cmd))
    subprocess.check_call(cmd)

def uninstall_incompatible_torchao():
    """Colab ships torchao 0.10; recent peft hard-raises unless torchao is absent or >=0.16.
    We do not use torchao quantization, so uninstalling is safer than upgrading Colab's stack.
    """
    try:
        import importlib.metadata as metadata
        version = metadata.version('torchao')
    except metadata.PackageNotFoundError:
        print('torchao not installed')
        return
    print(f'Removing incompatible torchao {version} (peft requires >=0.16 or absent)')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'],
        stdout=subprocess.DEVNULL,
    )
    for name in list(sys.modules):
        if name == 'torchao' or name.startswith('torchao.'):
            del sys.modules[name]
    try:
        from peft.import_utils import is_torchao_available
        is_torchao_available.cache_clear()
    except Exception:
        pass

torch = assert_colab_gpu()
repo_root = ensure_repo_root()
if repo_root in sys.path:
    sys.path.remove(repo_root)
sys.path.insert(0, repo_root)

pip_install('wildlife-datasets', 'timm', 'scikit-image', 'peft')
pip_install('git+https://github.com/WildlifeDatasets/wildlife-tools@main')
uninstall_incompatible_torchao()

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/SeaTurtle'
DATASET_DIRS = ('AmvrakikosTurtles', 'ReunionTurtles', 'ZakynthosTurtles')

def has_datasets(path):
    return all(os.path.isdir(os.path.join(path, name)) for name in DATASET_DIRS)

if not has_datasets(DRIVE_DATA):
    raise FileNotFoundError(
        f'Datasets not found at {DRIVE_DATA}. '
        'Put AmvrakikosTurtles, ReunionTurtles, ZakynthosTurtles in Drive > SeaTurtle.'
    )

device = torch.device('cuda')
root_features = os.path.join(DRIVE_DATA, 'features')
checkpoint_dir = os.path.join(DRIVE_DATA, 'checkpoints', 'lora_megadescriptor_opp')
os.makedirs(root_features, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

print(f'Repo: {repo_root}')
print(f'Data: {DRIVE_DATA} OK')
print(f'Features: {root_features}')
print(f'Checkpoints: {checkpoint_dir}')

Colab GPU: Tesla T4
Updating repo at /content/sides-matching to origin/main...
Repo commit: f812e9c
> /usr/bin/python3 -m pip install -q wildlife-datasets timm scikit-image peft
> /usr/bin/python3 -m pip install -q git+https://github.com/WildlifeDatasets/wildlife-tools@main
torchao not installed
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo: /content/sides-matching
Data: /content/drive/MyDrive/SeaTurtle OK
Features: /content/drive/MyDrive/SeaTurtle/features
Checkpoints: /content/drive/MyDrive/SeaTurtle/checkpoints/lora_megadescriptor_opp


In [16]:
import hashlib
import numpy as np
import pandas as pd
from sides_matching import amvrakikos, reunion_green, reunion_hawksbill, zakynthos
from sides_matching.train_lora import (
    TrainConfig,
    IdentityMapper,
    merge_train_dataframes,
    split_identities_stratified,
    set_seed,
    build_raw_train_datasets,
    build_source_validation_loaders,
    build_train_loader,
    build_hard_negative_map,
    mine_train_baseline_embeddings,
    ensure_orientation_column,
)

config = TrainConfig(
    loss_mode='triplet_hard',
    augment_policy='standard',
    checkpoint_metric='macro_opp_recall',
)
set_seed(config.seed)
print(
    f'Config: pair_batch={config.batch_size}, lora_r={config.lora_r}, '
    f'loss={config.loss_mode}, aug={config.augment_policy}, '
    f'opp_w={config.opposite_loss_weight}, amp={config.use_amp}, '
    f'grad_ckpt={config.grad_checkpointing}, epochs={config.epochs}'
)

root_data = DRIVE_DATA
train_sources = [
    ('Amvrakikos', os.path.join(root_data, 'AmvrakikosTurtles'), amvrakikos),
    ('ReunionGreen', os.path.join(root_data, 'ReunionTurtles'), reunion_green),
    ('ReunionHawksbill', os.path.join(root_data, 'ReunionTurtles'), reunion_hawksbill),
]

train_frames = []
for name, root, dataset_fn in train_sources:
    dataset = dataset_fn(root, transform=None)
    frame = ensure_orientation_column(dataset.df.copy())
    train_frames.append((name, frame))
    print(f'{name}: {len(frame)} images, {frame["identity"].nunique()} identities')

merged_df = merge_train_dataframes(train_frames)
split_fingerprint = hashlib.sha256(
    merged_df['global_identity'].sort_values().astype(str).str.cat(sep='|').encode()
).hexdigest()[:12]
train_df, val_df = split_identities_stratified(
    merged_df,
    val_fraction=config.val_identity_fraction,
    seed=config.seed,
)
identity_mapper = IdentityMapper.from_identities(train_df['global_identity'])
print(
    f'Train images: {len(train_df)} | Val images: {len(val_df)} | '
    f'ArcFace classes (train only): {identity_mapper.num_classes} | split={split_fingerprint}'
)

train_datasets, train_label_parts, train_orientation_parts = build_raw_train_datasets(
    train_sources, train_df, identity_mapper
)
negative_map = build_negative_map_for_training(
    config, train_sources, train_df, identity_mapper, device
)
train_loader = build_train_loader(
    config,
    train_datasets,
    train_label_parts,
    train_orientation_parts,
    negative_map=negative_map,
)
source_loaders, source_orientations, _ = build_source_validation_loaders(
    train_sources,
    val_df,
    config.img_size,
    config.batch_size,
    config.num_workers,
)
print(
    f'Triplet train loader ready | source val loaders={list(source_loaders)} | '
    f'checkpoint metric={config.checkpoint_metric}'
)

zakynthos_root = os.path.join(root_data, 'ZakynthosTurtles')
zakynthos_dataset = zakynthos(zakynthos_root, transform=None)
print(f'Zakynthos (test only, sealed): {len(zakynthos_dataset.df)} images, {zakynthos_dataset.df["identity"].nunique()} identities')

Config: pair_batch=1, lora_r=16, opp_w=1.0, amp=True, grad_ckpt=True, epochs=40
Amvrakikos: 200 images, 50 identities
ReunionGreen: 200 images, 50 identities
ReunionHawksbill: 136 images, 34 identities
Train images: 428 | Val images: 108 | Classes: 134
Opposite-pair train loader ready | val images=108 | both-side IDs emphasized during training
Zakynthos (test only): 160 images, 40 identities


In [17]:
from sides_matching.train_lora import (
    build_training_model,
    configure_optimizer,
    clear_cuda_memory,
)

for _name in ('model', 'optimizer', 'scaler', 'baseline_model', 'lora_model'):
    if _name in globals():
        del globals()[_name]
clear_cuda_memory()

model = build_training_model(identity_mapper.num_classes, config, device)
optimizer = configure_optimizer(model, config)
scaler = torch.cuda.amp.GradScaler(enabled=config.use_amp and device.type == 'cuda')
model.backbone.print_trainable_parameters()
print(
    f'Train memory settings: batch_size={config.batch_size}, '
    f'lora_r={config.lora_r}, amp={config.use_amp}, '
    f'grad_checkpointing={config.grad_checkpointing}'
)
print(f'GPU mem allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

Gradient checkpointing: on
trainable params: 4,624,128 || all params: 199,822,644 || trainable%: 2.3141
Train memory settings: batch_size=1, lora_r=16, amp=True, grad_checkpointing=True
GPU mem allocated: 0.84 GB


/tmp/ipykernel_8967/1897912378.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=config.use_amp and device.type == 'cuda')


In [18]:
from sides_matching.train_lora import (
    run_training_loop,
    clear_cuda_memory,
    is_cuda_oom,
    build_train_loader,
)

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
metrics_csv = os.path.join(checkpoint_dir, 'training_metrics.csv')

def rebuild_loaders():
    global train_loader
    train_loader = build_train_loader(
        config,
        train_datasets,
        train_label_parts,
        train_orientation_parts,
        negative_map=negative_map,
    )

while True:
    try:
        history_df = run_training_loop(
            model,
            optimizer,
            train_loader,
            source_loaders,
            source_orientations,
            identity_mapper,
            config,
            device,
            checkpoint_dir,
            scaler=scaler,
            metrics_csv=metrics_csv,
        )
        break
    except Exception as exc:
        if not is_cuda_oom(exc):
            raise
        clear_cuda_memory()
        if config.batch_size <= 1:
            raise RuntimeError(
                'CUDA OOM even at pair_batch=1. Restart runtime and re-run from cell 1.'
            ) from exc
        config.batch_size = max(1, config.batch_size // 2)
        rebuild_loaders()
        print(f'CUDA OOM: reduced pair_batch to {config.batch_size}, retrying training...')

display(history_df.tail())
print(f'Metrics log: {metrics_csv}')

Epoch 01 | train_loss=17.8158 | val_loss=22.3778 | embed@1=0.7500 | opp@1=0.7315 | pair_batch=1
Saved best opposite-side checkpoint to /content/drive/MyDrive/SeaTurtle/checkpoints/lora_megadescriptor_opp
Epoch 02 | train_loss=12.5300 | val_loss=23.4078 | embed@1=0.7685 | opp@1=0.7685 | pair_batch=1
Saved best opposite-side checkpoint to /content/drive/MyDrive/SeaTurtle/checkpoints/lora_megadescriptor_opp
Epoch 03 | train_loss=7.0686 | val_loss=24.4341 | embed@1=0.7500 | opp@1=0.7593 | pair_batch=1
Epoch 04 | train_loss=3.7457 | val_loss=25.3835 | embed@1=0.7963 | opp@1=0.7500 | pair_batch=1
Epoch 05 | train_loss=1.8963 | val_loss=26.4161 | embed@1=0.7593 | opp@1=0.7407 | pair_batch=1
Epoch 06 | train_loss=1.3608 | val_loss=27.0327 | embed@1=0.7407 | opp@1=0.7315 | pair_batch=1
Epoch 07 | train_loss=1.2717 | val_loss=27.3328 | embed@1=0.7593 | opp@1=0.7407 | pair_batch=1
Epoch 08 | train_loss=0.9610 | val_loss=27.4680 | embed@1=0.7500 | opp@1=0.7222 | pair_batch=1
Epoch 09 | train_loss=

,epoch,train_loss,val_loss,val_recall,embed_recall,opp_recall,batch_size
7,8,0.961008,27.467957,0.0,0.750000,0.722222,1
8,9,0.887720,27.971086,0.0,0.759259,0.722222,1
9,10,0.749754,28.445073,0.0,0.750000,0.722222,1
10,11,0.592044,28.688649,0.0,0.768519,0.712963,1
11,12,0.567790,28.394315,0.0,0.768519,0.722222,1


In [19]:
import pickle
from sides_matching import get_features
from sides_matching.train_lora import build_inference_model, extract_features, save_feature_pickle, get_eval_transform, clear_cuda_memory
from sides_matching.evaluation import verify_adapter_active
from wildlife_tools.features import DeepFeatures
import timm

adapter_dir = os.path.join(checkpoint_dir, 'adapter')
if not os.path.isdir(adapter_dir):
    raise FileNotFoundError(f'Missing adapter at {adapter_dir}. Run training first.')

for _name in ('model', 'optimizer', 'scaler'):
    if _name in globals():
        del globals()[_name]
clear_cuda_memory()

lora_model = build_inference_model(config, adapter_dir, device)
grayscale = False
extract_batch = max(1, min(config.batch_size, 4))

# Adapter verification on one Zakynthos batch (same preprocessing as eval).
probe_dataset = zakynthos(zakynthos_root, transform=get_eval_transform(flip=False, img_size=config.img_size))
probe_loader = torch.utils.data.DataLoader(probe_dataset, batch_size=min(4, len(probe_dataset)), shuffle=False)
probe_batch = next(iter(probe_loader))
if isinstance(probe_batch, (tuple, list)):
    probe_batch = probe_batch[0]
active, max_diff = verify_adapter_active(lora_model, probe_batch)
print(f'Adapter verification: active={active}, max_diff={max_diff:.6f}')
if not active:
    raise RuntimeError('LoRA adapter did not change embeddings — check checkpoint loading.')

for flip in [True, False]:
    transform = get_eval_transform(flip=flip, img_size=config.img_size)
    eval_dataset = zakynthos(zakynthos_root, transform=transform)
    features = extract_features(lora_model, eval_dataset, device, batch_size=extract_batch)
    file_name = os.path.join(
        root_features,
        f'MegaDescriptorLoRA_Zakynthos_flip={flip}_grayscale={grayscale}.pickle',
    )
    save_feature_pickle(features, file_name)
    print(f'Saved {file_name} shape={features.shape}')

del lora_model
clear_cuda_memory()

baseline_model = timm.create_model(config.model_name, num_classes=0, pretrained=True).to(device)
baseline_model.eval()
baseline_extractor = DeepFeatures(baseline_model, batch_size=extract_batch, device=device)

for flip in [True, False]:
    transform = get_eval_transform(flip=flip, img_size=config.img_size)
    eval_dataset = zakynthos(zakynthos_root, transform=transform)
    file_name = os.path.join(
        root_features,
        f'MegaDescriptor_Zakynthos_flip={flip}_grayscale={grayscale}.pickle',
    )
    if not os.path.exists(file_name):
        get_features(file_name, eval_dataset, baseline_extractor)
        print(f'Extracted baseline {file_name}')
    else:
        print(f'Using existing baseline {file_name}')

# Embedding drift diagnostic (flip=False).
def _load_feats(path):
    obj = pickle.load(open(path, 'rb'))
    return obj.features if hasattr(obj, 'features') else obj

base_path = os.path.join(root_features, 'MegaDescriptor_Zakynthos_flip=False_grayscale=False.pickle')
lora_path = os.path.join(root_features, 'MegaDescriptorLoRA_Zakynthos_flip=False_grayscale=False.pickle')
base_feats = _load_feats(base_path)
lora_feats = _load_feats(lora_path)
cos_diag = (base_feats * lora_feats).sum(axis=1) / (
    np.linalg.norm(base_feats, axis=1) * np.linalg.norm(lora_feats, axis=1)
)
print(f'Mean per-image cos(base, lora): {cos_diag.mean():.4f}')

del baseline_model, baseline_extractor
clear_cuda_memory()
print('Feature extraction done for base + LoRA.')

Saved /content/drive/MyDrive/SeaTurtle/features/MegaDescriptorLoRA_Zakynthos_flip=True_grayscale=False.pickle shape=(160, 1536)
Saved /content/drive/MyDrive/SeaTurtle/features/MegaDescriptorLoRA_Zakynthos_flip=False_grayscale=False.pickle shape=(160, 1536)
Using existing baseline /content/drive/MyDrive/SeaTurtle/features/MegaDescriptor_Zakynthos_flip=True_grayscale=False.pickle
Using existing baseline /content/drive/MyDrive/SeaTurtle/features/MegaDescriptor_Zakynthos_flip=False_grayscale=False.pickle
Feature extraction done for base + LoRA.


In [20]:
import importlib
import pickle
import sides_matching.predictions as _predictions
import sides_matching.train_lora as _train_lora

if 'ensure_repo_root' in globals():
    ensure_repo_root()
for _name in list(sys.modules):
    if _name == 'sides_matching' or _name.startswith('sides_matching.'):
        del sys.modules[_name]
importlib.invalidate_caches()

from sides_matching.train_lora import compare_base_vs_lora
from sides_matching.evaluation import compare_feature_opposite_bootstrap

mods = [
    'full',
    'same orientation',
    'different orientation',
    'same year',
    'different year',
    'different both',
]

results_df, comparison_df = compare_base_vs_lora(
    zakynthos_dataset.df,
    root_features,
    flips=(True, False),
    grayscale=False,
    mods=mods,
)

print('Base MegaDescriptor vs LoRA fine-tune on held-out Zakynthos')
print('delta = LoRA - base (positive means LoRA is better)')
display(comparison_df.round(4))

focus = comparison_df[comparison_df['mod'] == 'different orientation'].copy()
print('\nOpposite-side (different orientation) Top-1')
display(focus[['flip', 'base_top1', 'lora_top1', 'delta_top1']].round(4))

# Identity-clustered bootstrap on primary endpoint (flip=False, different orientation).
def _load_feats(path):
    obj = pickle.load(open(path, 'rb'))
    return obj.features if hasattr(obj, 'features') else obj

base_feats = _load_feats(os.path.join(root_features, 'MegaDescriptor_Zakynthos_flip=False_grayscale=False.pickle'))
lora_feats = _load_feats(os.path.join(root_features, 'MegaDescriptorLoRA_Zakynthos_flip=False_grayscale=False.pickle'))
df_eval = zakynthos_dataset.df.reset_index(drop=True)
bootstrap = compare_feature_opposite_bootstrap(
    base_feats,
    lora_feats,
    df_eval['identity'].to_numpy(),
    df_eval['orientation'].to_numpy(),
    n_bootstrap=2000,
    seed=config.seed,
)
print(
    f'\nPrimary endpoint bootstrap (flip=False, different orientation): '
    f'mean_delta={bootstrap.mean_delta:.4f}, '
    f'95% CI=[{bootstrap.ci_low:.4f}, {bootstrap.ci_high:.4f}]'
)
if bootstrap.mean_delta > 0 and bootstrap.ci_low > 0:
    print('Result: statistically positive improvement (CI excludes zero).')
elif bootstrap.ci_high < 0:
    print('Result: statistically negative (LoRA worse than base).')
else:
    print('Result: inconclusive (CI overlaps zero).')

results_csv = os.path.join(checkpoint_dir, 'zakynthos_eval.csv')
comparison_csv = os.path.join(checkpoint_dir, 'zakynthos_base_vs_lora.csv')
bootstrap_csv = os.path.join(checkpoint_dir, 'zakynthos_bootstrap.csv')
results_df.to_csv(results_csv, index=False)
comparison_df.to_csv(comparison_csv, index=False)
pd.DataFrame([{
    'mean_delta': bootstrap.mean_delta,
    'ci_low': bootstrap.ci_low,
    'ci_high': bootstrap.ci_high,
    'n_bootstrap': bootstrap.n_bootstrap,
}]).to_csv(bootstrap_csv, index=False)
print(f'Saved {results_csv}')
print(f'Saved {comparison_csv}')
print(f'Saved {bootstrap_csv}')

Updating repo at /content/sides-matching to origin/main...
Repo commit: f812e9c
Base MegaDescriptor vs LoRA fine-tune on held-out Zakynthos
delta = LoRA - base (positive means LoRA is better)


,flip,mod,base_top1,lora_top1,delta_top1,base_top5,lora_top5,delta_top5
0,False,different both,0.3125,0.3125,0.0000,0.6937,0.6625,-0.0312
1,False,different orientation,0.6250,0.6312,0.0062,0.9500,0.9562,0.0062
2,False,different year,0.5937,0.6062,0.0125,0.9125,0.9000,-0.0125
3,False,full,0.7562,0.7812,0.0250,0.9875,0.9875,0.0000
4,False,same orientation,0.5312,0.5500,0.0187,0.8625,0.8375,-0.0250
5,False,same year,0.5750,0.5562,-0.0187,0.9187,0.9125,-0.0062
6,True,different both,0.3125,0.3062,-0.0062,0.6937,0.6750,-0.0187
7,True,different orientation,0.6125,0.5937,-0.0187,0.9375,0.9500,0.0125
8,True,different year,0.5875,0.5937,0.0062,0.9000,0.8937,-0.0062
9,True,full,0.7625,0.7562,-0.0062,0.9750,0.9750,0.0000



Opposite-side (different orientation) Top-1


,flip,base_top1,lora_top1,delta_top1
1,False,0.6250,0.6312,0.0062
7,True,0.6125,0.5937,-0.0187


Saved /content/drive/MyDrive/SeaTurtle/checkpoints/lora_megadescriptor_opp/zakynthos_eval.csv
Saved /content/drive/MyDrive/SeaTurtle/checkpoints/lora_megadescriptor_opp/zakynthos_base_vs_lora.csv


In [ ]:
# Source-only ablation smoke test (5 epochs). Set True to compare recipes before full training.
RUN_ABLATION = False
ABLATION_EPOCHS = 5

if RUN_ABLATION:
    from dataclasses import replace
    from sides_matching.train_lora import run_ablation_smoke, TrainConfig

    ablation_root = os.path.join(DRIVE_DATA, 'checkpoints', 'lora_ablations')
    ablation_csv = os.path.join(ablation_root, 'ablation_smoke.csv')
    recipes = [
        ('pair_baseline', replace(config, loss_mode='pair', augment_policy='minimal')),
        ('triplet_random', replace(config, loss_mode='triplet', augment_policy='minimal')),
        ('triplet_hard', replace(config, loss_mode='triplet_hard', augment_policy='standard')),
    ]
    holdout_sets = [
        ('Amvrakikos', ('Amvrakikos',)),
        ('Reunion', ('ReunionGreen', 'ReunionHawksbill')),
    ]
    ablation_rows = []
    for recipe_name, recipe_config in recipes:
        for holdout_name, holdout_sources in holdout_sets:
            metrics = run_ablation_smoke(
                recipe_name=f'{recipe_name}_{holdout_name}',
                config=replace(recipe_config, seed=42),
                train_sources=train_sources,
                merged_df=merged_df,
                holdout_sources=holdout_sources,
                device=device,
                checkpoint_root=ablation_root,
                epochs=ABLATION_EPOCHS,
                metrics_csv=ablation_csv,
            )
            ablation_rows.append({'recipe': recipe_name, 'holdout': holdout_name, **metrics})
            print(f'{recipe_name} / holdout={holdout_name}: {metrics}')
    display(pd.DataFrame(ablation_rows))
    print(f'Ablation log: {ablation_csv}')
else:
    print('Skipping ablation smoke (RUN_ABLATION=False).')

In [ ]:
# Optional multi-seed final training (source-only checkpoint selection).
RUN_MULTI_SEED = False
FINAL_SEEDS = [17, 42, 83, 101, 211]

if RUN_MULTI_SEED:
    from dataclasses import replace
    from sides_matching.train_lora import (
        set_seed,
        build_training_model,
        configure_optimizer,
        build_raw_train_datasets,
        build_source_validation_loaders,
        build_train_loader,
        build_hard_negative_map,
        mine_train_baseline_embeddings,
        run_training_loop,
        IdentityMapper,
        split_identities_stratified,
        clear_cuda_memory,
    )

    multi_seed_root = os.path.join(DRIVE_DATA, 'checkpoints', 'lora_multiseed')
    seed_summaries = []
    for seed in FINAL_SEEDS:
        seed_config = replace(config, seed=seed)
        set_seed(seed)
        seed_dir = os.path.join(multi_seed_root, f'seed_{seed}')
        train_df_s, val_df_s = split_identities_stratified(
            merged_df, val_fraction=seed_config.val_identity_fraction, seed=seed
        )
        mapper_s = IdentityMapper.from_identities(train_df_s['global_identity'])
        datasets_s, labels_s, orients_s = build_raw_train_datasets(train_sources, train_df_s, mapper_s)
        emb_s, lab_s = mine_train_baseline_embeddings(train_sources, train_df_s, mapper_s, device, seed_config)
        neg_s = build_hard_negative_map(emb_s, lab_s, seed=seed, top_k=seed_config.hard_negative_top_k)
        loader_s = build_train_loader(seed_config, datasets_s, labels_s, orients_s, negative_map=neg_s)
        src_loaders_s, src_orients_s, _ = build_source_validation_loaders(
            train_sources, val_df_s, seed_config.img_size, seed_config.batch_size, seed_config.num_workers
        )
        model_s = build_training_model(mapper_s.num_classes, seed_config, device)
        opt_s = configure_optimizer(model_s, seed_config)
        scaler_s = torch.cuda.amp.GradScaler(enabled=seed_config.use_amp and device.type == 'cuda')
        hist_s = run_training_loop(
            model_s, opt_s, loader_s, src_loaders_s, src_orients_s, mapper_s,
            seed_config, device, seed_dir, scaler=scaler_s,
            metrics_csv=os.path.join(seed_dir, 'training_metrics.csv'),
        )
        seed_summaries.append({
            'seed': seed,
            'best_macro_opp': float(hist_s['macro_opp_recall'].max()),
            'checkpoint_dir': seed_dir,
        })
        del model_s, opt_s, scaler_s
        clear_cuda_memory()
    display(pd.DataFrame(seed_summaries))
else:
    print('Skipping multi-seed training (RUN_MULTI_SEED=False).')

In [21]:
# Confirm trained model is on Google Drive (and zip for easy download/share)
import glob

required = [
    os.path.join(checkpoint_dir, 'adapter'),
    os.path.join(checkpoint_dir, 'arcface_head.pt'),
    os.path.join(checkpoint_dir, 'train_config.json'),
]
missing = [path for path in required if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(
        'Training checkpoint incomplete. Missing:\n  - ' + '\n  - '.join(missing)
    )

adapter_files = sorted(glob.glob(os.path.join(checkpoint_dir, 'adapter', '*')))
print(f'Checkpoint OK at {checkpoint_dir}')
for path in required:
    print(f'  - {path}')
print(f'  - adapter files ({len(adapter_files)}):')
for path in adapter_files:
    print(f'      {os.path.basename(path)}')

zip_base = os.path.join(DRIVE_DATA, 'checkpoints', 'lora_megadescriptor_opp_bundle')
shutil.make_archive(zip_base, 'zip', checkpoint_dir)
print(f'\nZipped model -> {zip_base}.zip')
print('Open: drive.google.com → MyDrive → SeaTurtle → checkpoints/')

Checkpoint OK at /content/drive/MyDrive/SeaTurtle/checkpoints/lora_megadescriptor_opp
  - /content/drive/MyDrive/SeaTurtle/checkpoints/lora_megadescriptor_opp/adapter
  - /content/drive/MyDrive/SeaTurtle/checkpoints/lora_megadescriptor_opp/arcface_head.pt
  - /content/drive/MyDrive/SeaTurtle/checkpoints/lora_megadescriptor_opp/train_config.json
  - adapter files (3):
      README.md
      adapter_config.json
      adapter_model.safetensors

Zipped model -> /content/drive/MyDrive/SeaTurtle/checkpoints/lora_megadescriptor_opp_bundle.zip
Open: drive.google.com → MyDrive → SeaTurtle → checkpoints/
